## News Impact Curve - GJR GARCH

In [ ]:
from arch import arch_model
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

# --- your residuals here ---
# residuals = actual - chronos_forecast

# ── 1. Fit standard GARCH(1,1) ──────────────────────────────────────────────
garch = arch_model(residuals_clean, vol='Garch', p=1, o=0, q=1, dist='normal')
garch_fit = garch.fit(disp='off')
print(garch_fit.summary())

std_resid = garch_fit.std_resid  # standardized residuals

# ── 2. News Impact Curve ─────────────────────────────────────────────────────
omega = garch_fit.params['omega']
alpha = garch_fit.params['alpha[1]']
beta  = garch_fit.params['beta[1]']
avg_h = garch_fit.conditional_volatility.mean() ** 2  # unconditional variance proxy

shock_range = np.linspace(-3 * std_resid.std(), 3 * std_resid.std(), 500)
nic = omega + alpha * shock_range**2 + beta * avg_h  # h_{t+1} as function of epsilon_t

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(shock_range, nic, color='steelblue', linewidth=2)
axes[0].axvline(0, color='gray', linestyle='--', linewidth=0.8)
axes[0].set_title("News Impact Curve (Standard GARCH)")
axes[0].set_xlabel("Shock (ε)")
axes[0].set_ylabel("Conditional Variance h(t+1)")

# ── 3. Sign Bias Test (Engle & Ng, 1993) ────────────────────────────────────
e = std_resid.values[:-1]          # lagged standardized residuals
sq_next = std_resid.values[1:]**2  # squared std resid at t

S_neg = (e < 0).astype(float)      # indicator: negative shock
S_pos = (e >= 0).astype(float)     # indicator: positive shock

# Regressors: const, S-, S-*e, S+*e
X = np.column_stack([
    np.ones(len(e)),
    S_neg,
    S_neg * e,
    S_pos * e
])

# OLS
result = np.linalg.lstsq(X, sq_next, rcond=None)
coef = result[0]
fitted = X @ coef
resid_ols = sq_next - fitted
n, k = X.shape
se = np.sqrt(np.diag(np.linalg.inv(X.T @ X) * (resid_ols @ resid_ols) / (n - k)))
t_stats = coef / se
p_values = 2 * stats.t.sf(np.abs(t_stats), df=n - k)

labels = ['Const', 'Sign Bias (S⁻)', 'Neg Size Bias (S⁻·ε)', 'Pos Size Bias (S⁺·ε)']
print("\n── Sign Bias Test (Engle & Ng, 1993) ──")
print(f"{'Term':<25} {'Coef':>10} {'t-stat':>10} {'p-value':>10}")
print("-" * 58)
for i in range(k):
    sig = " *" if p_values[i] < 0.05 else ""
    print(f"{labels[i]:<25} {coef[i]:>10.4f} {t_stats[i]:>10.4f} {p_values[i]:>10.4f}{sig}")

# Joint F-test on the 3 bias terms (indices 1,2,3)
R = X[:, 1:]  # drop const
coef_bias = coef[1:]
fitted_r = R @ coef_bias
SS_res = resid_ols @ resid_ols
SS_reg = fitted_r @ fitted_r
F_stat = (SS_reg / 3) / (SS_res / (n - k))
F_pval = stats.f.sf(F_stat, 3, n - k)
print(f"\nJoint F-test (3 bias terms): F = {F_stat:.4f}, p = {F_pval:.4f}")

# ── 4. Decision ──────────────────────────────────────────────────────────────
sign_bias_sig   = p_values[1] < 0.05
neg_size_sig    = p_values[2] < 0.05
pos_size_sig    = p_values[3] < 0.05
joint_sig       = F_pval < 0.05

print("\n── Decision ──")
if joint_sig or sign_bias_sig:
    print("→ Asymmetry detected → use GJR-GARCH (or EGARCH)")
else:
    print("→ No significant asymmetry → standard GARCH is sufficient")

# ── 5. Visualize sign bias ───────────────────────────────────────────────────
neg_idx = e < 0
pos_idx = e >= 0
axes[1].scatter(e[neg_idx], sq_next[neg_idx], alpha=0.3, s=10, color='tomato',  label='Negative shocks')
axes[1].scatter(e[pos_idx], sq_next[pos_idx], alpha=0.3, s=10, color='steelblue', label='Positive shocks')
axes[1].set_title("Shock vs Squared Std Residual\n(Sign Bias Visualization)")
axes[1].set_xlabel("Lagged Std Residual (ε)")
axes[1].set_ylabel("Squared Std Residual (ε²)")
axes[1].legend()

plt.tight_layout()
plt.show()

## IGARCH

In [ ]:
from arch import arch_model
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss

# --- your residuals here ---
# residuals = actual - chronos_forecast

# ── 1. Fit GARCH(1,1) ────────────────────────────────────────────────────────
garch = arch_model(residuals_clean, vol='Garch', p=1, o=0, q=1, dist='normal')
garch_fit = garch.fit(disp='off')

alpha = garch_fit.params['alpha[1]']
beta  = garch_fit.params['beta[1]']
persistence = alpha + beta

print("── GARCH(1,1) Persistence ──")
print(f"α̂          : {alpha:.4f}")
print(f"β̂          : {beta:.4f}")
print(f"α̂ + β̂      : {persistence:.4f}")

if persistence >= 0.99:
    print("→ IGARCH territory: variance shocks are permanent")
elif persistence >= 0.95:
    print("→ High persistence: very slow mean reversion")
else:
    print("→ Moderate persistence: variance mean-reverts")

# ── 2. Half-life of variance shocks ─────────────────────────────────────────
if persistence < 1.0:
    half_life = np.log(0.5) / np.log(persistence)
    print(f"\n── Half-life of Variance Shocks ──")
    print(f"Half-life  : {half_life:.2f} periods")
    print(f"Interpretation: variance takes ~{half_life:.1f} periods to revert halfway to mean")
else:
    print("\n→ Persistence = 1.0 → half-life is infinite (IGARCH)")

# ── 3. Unit Root Tests on Squared Residuals ──────────────────────────────────
sq_resid = residuals_clean ** 2

# ADF — null: unit root (non-stationary)
adf_stat, adf_p, _, _, adf_crit, _ = adfuller(sq_resid, autolag='AIC')

# KPSS — null: stationary
kpss_stat, kpss_p, _, kpss_crit = kpss(sq_resid, regression='c', nlags='auto')

print("\n── Unit Root Tests on Squared Residuals ──")
print(f"ADF  stat: {adf_stat:.4f}  p-value: {adf_p:.4f}  → {'stationary' if adf_p < 0.05 else 'unit root present'}")
print(f"KPSS stat: {kpss_stat:.4f}  p-value: {kpss_p:.4f}  → {'unit root present' if kpss_p < 0.05 else 'stationary'}")

print("\n── Combined Decision ──")
if persistence >= 0.99:
    print("→ High persistence + possible unit root → consider IGARCH or FIGARCH")
elif persistence >= 0.95 and adf_p > 0.05:
    print("→ Near-unit-root persistence → treat forecasts with wide uncertainty bounds")
else:
    print("→ Variance is stationary → standard GARCH is appropriate")

## FIGARCH

In [ ]:
from arch import arch_model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf
from scipy import stats

# --- your residuals here ---
# residuals = actual - chronos_forecast

sq_resid  = np.array(residuals_clean) ** 2
abs_resid = np.abs(residuals_clean)

# ── 1. ACF of |residuals| and residuals² ─────────────────────────────────────
n_lags = 60
acf_sq  = acf(sq_resid,  nlags=n_lags, fft=True)
acf_abs = acf(abs_resid, nlags=n_lags, fft=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].stem(range(n_lags+1), acf_sq,  linefmt='steelblue', markerfmt='o', basefmt='gray')
axes[0].set_title("ACF of Squared Residuals")
axes[0].set_xlabel("Lag")
axes[1].stem(range(n_lags+1), acf_abs, linefmt='tomato',    markerfmt='o', basefmt='gray')
axes[1].set_title("ACF of |Residuals|")
axes[1].set_xlabel("Lag")
plt.tight_layout()
plt.show()

# ── 2. Hyperbolic vs Exponential Decay Check ─────────────────────────────────
# Fit log(ACF) ~ log(lag) — if slope is between -1 and 0 → hyperbolic (long memory)
# Fit log(ACF) ~ lag     — if linear → exponential (short memory)
lags     = np.arange(1, n_lags + 1)
acf_vals = acf_sq[1:]  # drop lag 0

pos_mask = acf_vals > 0  # log only defined for positive values
log_lags = np.log(lags[pos_mask])
log_acf  = np.log(acf_vals[pos_mask])

# Log-log fit (hyperbolic)
slope_hyp, intercept_hyp, r_hyp, _, _ = stats.linregress(log_lags, log_acf)

# Log-linear fit (exponential)
slope_exp, intercept_exp, r_exp, _, _ = stats.linregress(lags[pos_mask], log_acf)

print("── Decay Pattern Analysis ──")
print(f"Log-log  fit  (hyperbolic): slope = {slope_hyp:.4f},  R² = {r_hyp**2:.4f}")
print(f"Log-linear fit (exponential): slope = {slope_exp:.4f}, R² = {r_exp**2:.4f}")
if r_hyp**2 > r_exp**2 and -1 < slope_hyp < 0:
    print("→ Hyperbolic decay detected → long memory present")
else:
    print("→ Exponential decay → short memory, standard GARCH sufficient")

# ── 3. GPH Test (Geweke-Porter-Hudak) ────────────────────────────────────────
# Estimates fractional differencing parameter d via log-periodogram regression
def gph_test(series, bandwidth_exp=0.5):
    n = len(series)
    m = max(2, int(n ** bandwidth_exp))  # bandwidth

    # Periodogram
    freq  = np.fft.rfftfreq(n)[1:m+1]
    pgram = np.abs(np.fft.rfft(series - series.mean()))[1:m+1] ** 2

    # GPH regression: log(I(w)) = c - d*log(4*sin²(w/2)) + e
    x = np.log(4 * np.sin(np.pi * freq) ** 2)
    y = np.log(pgram + 1e-10)

    slope, intercept, r, p, se = stats.linregress(x, y)
    d_hat = -slope  # d estimate

    t_stat = d_hat / se
    p_val  = 2 * stats.t.sf(abs(t_stat), df=m - 2)

    return d_hat, se, t_stat, p_val

d_sq,  se_sq,  t_sq,  p_sq  = gph_test(sq_resid)
d_abs, se_abs, t_abs, p_abs = gph_test(abs_resid)

print("\n── GPH Long Memory Test ──")
print(f"{'Series':<20} {'d̂':>8} {'SE':>8} {'t-stat':>8} {'p-value':>8}")
print("-" * 56)
print(f"{'Squared residuals':<20} {d_sq:>8.4f} {se_sq:>8.4f} {t_sq:>8.4f} {p_sq:>8.4f}", "  *" if p_sq < 0.05 else "")
print(f"{'|Residuals|':<20} {d_abs:>8.4f} {se_abs:>8.4f} {t_abs:>8.4f} {p_abs:>8.4f}", "  *" if p_abs < 0.05 else "")

# ── 4. R/S (Hurst Exponent) ──────────────────────────────────────────────────
def hurst_rs(series, min_chunk=8):
    n      = len(series)
    chunks = [n // k for k in range(2, int(np.log2(n))) if n // k >= min_chunk]
    rs_vals, ns = [], []
    for chunk in set(chunks):
        segments = [series[i:i+chunk] for i in range(0, n - chunk + 1, chunk)]
        rs_seg = []
        for seg in segments:
            mean_s = np.mean(seg)
            dev    = np.cumsum(seg - mean_s)
            R      = dev.max() - dev.min()
            S      = np.std(seg, ddof=1)
            if S > 0:
                rs_seg.append(R / S)
        if rs_seg:
            rs_vals.append(np.mean(rs_seg))
            ns.append(chunk)

    log_n  = np.log(ns)
    log_rs = np.log(rs_vals)
    H, _, _, _, _ = stats.linregress(log_n, log_rs)
    return H

H_sq  = hurst_rs(sq_resid)
H_abs = hurst_rs(abs_resid)

print("\n── Hurst Exponent (R/S Analysis) ──")
print(f"Squared residuals : H = {H_sq:.4f}")
print(f"|Residuals|        : H = {H_abs:.4f}")
print("  H > 0.5 → long memory (persistent)")
print("  H = 0.5 → random walk")
print("  H < 0.5 → anti-persistent")

# ── 5. Final Decision ────────────────────────────────────────────────────────
long_memory_flags = [
    p_sq  < 0.05 and d_sq  > 0,
    p_abs < 0.05 and d_abs > 0,
    H_sq  > 0.55,
    r_hyp**2 > r_exp**2 and -1 < slope_hyp < 0
]

print("\n── Final Decision ──")
if sum(long_memory_flags) >= 2:
    print("→ Long memory confirmed → consider FIGARCH or HYGARCH")
elif sum(long_memory_flags) == 1:
    print("→ Weak evidence of long memory → proceed with caution, GARCH may still be okay")
else:
    print("→ No long memory → standard GARCH / GJR-GARCH is appropriate")

In [ ]:
from arch import arch_model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from itertools import combinations

# --- your residuals here ---
# residuals = actual - chronos_forecast
resid_array = np.array(residuals)

# ── 1. Fit GARCH(1,1) and extract standardized residuals ─────────────────────
garch = arch_model(residuals, vol='Garch', p=1, o=0, q=1, dist='normal')
garch_fit = garch.fit(disp='off')
std_resid = garch_fit.std_resid.dropna().values

# ── 2. Excess Kurtosis of GARCH Residuals ────────────────────────────────────
kurt       = stats.kurtosis(std_resid, fisher=True)  # excess kurtosis (normal=0)
_, jb_p    = stats.jarque_bera(std_resid)
_, sw_p    = stats.shapiro(std_resid[:5000])          # Shapiro-Wilk (max 5000)

print("── Excess Kurtosis Check ──")
print(f"Excess kurtosis : {kurt:.4f}  (normal = 0)")
print(f"Jarque-Bera p   : {jb_p:.4f}  {'→ non-normal *' if jb_p < 0.05 else '→ normal'}")
print(f"Shapiro-Wilk p  : {sw_p:.4f}  {'→ non-normal *' if sw_p < 0.05 else '→ normal'}")

if kurt > 1.0:
    print("→ Heavy tails remain → refit GARCH with t or skewed-t distribution")
elif kurt > 0:
    print("→ Mild excess kurtosis → normal dist may be acceptable")
else:
    print("→ No excess kurtosis → normal distribution is fine")

# ── 3. Chow Test ─────────────────────────────────────────────────────────────
def chow_test(series, break_point):
    n  = len(series)
    y1 = series[:break_point]
    y2 = series[break_point:]

    def sse(y):
        return np.sum((y - y.mean()) ** 2)

    sse_pooled   = sse(series)
    sse_split    = sse(y1) + sse(y2)
    k            = 1  # number of parameters (mean)
    F_stat       = ((sse_pooled - sse_split) / k) / (sse_split / (n - 2 * k))
    p_val        = stats.f.sf(F_stat, k, n - 2 * k)
    return F_stat, p_val

# Test at every candidate break point (trim 15% from each end)
n         = len(resid_array)
trim      = int(0.15 * n)
candidates = range(trim, n - trim)

chow_results = [(bp, *chow_test(resid_array, bp)) for bp in candidates]
chow_df      = pd.DataFrame(chow_results, columns=['break_point', 'F_stat', 'p_value'])
best_chow    = chow_df.loc[chow_df['F_stat'].idxmax()]

print("\n── Chow Test ──")
print(f"Best break point : index {int(best_chow.break_point)}")
print(f"F-statistic      : {best_chow.F_stat:.4f}")
print(f"p-value          : {best_chow.p_value:.4f}  {'→ structural break detected *' if best_chow.p_value < 0.05 else '→ no break'}")

# ── 4. Bai-Perron (sequential breakpoint detection) ──────────────────────────
def bai_perron(series, max_breaks=5, min_segment=10):
    n      = len(series)
    breaks = []
    search = list(range(min_segment, n - min_segment))

    for _ in range(max_breaks):
        best_bp, best_rss = None, np.inf
        endpoints = [0] + breaks + [n]

        for bp in search:
            # Skip if too close to existing breaks
            if any(abs(bp - b) < min_segment for b in breaks + [0, n]):
                continue
            seg_breaks = sorted(breaks + [bp])
            all_segs   = [0] + seg_breaks + [n]
            rss = sum(
                np.sum((series[all_segs[i]:all_segs[i+1]] -
                        series[all_segs[i]:all_segs[i+1]].mean()) ** 2)
                for i in range(len(all_segs) - 1)
            )
            if rss < best_rss:
                best_rss, best_bp = rss, bp

        if best_bp is None:
            break

        # Significance check via F-test
        seg_no_break = series[sorted([0] + breaks + [n])[
            next(i for i, b in enumerate([0] + breaks + [n]) if b > best_bp) - 1
        ]:sorted([0] + breaks + [n])[
            next(i for i, b in enumerate([0] + breaks + [n]) if b > best_bp)
        ]]
        _, p = chow_test(series, best_bp - (best_bp % 1))  # reuse chow for p-value
        if p < 0.05:
            breaks.append(best_bp)
        else:
            break

    return sorted(breaks)

bp_list = bai_perron(resid_array)

print("\n── Bai-Perron Structural Break Test ──")
if bp_list:
    print(f"Break points detected at indices: {bp_list}")
    for bp in bp_list:
        print(f"  Index {bp}: segment mean before = {resid_array[:bp].mean():.4f}, after = {resid_array[bp:].mean():.4f}")
else:
    print("No significant structural breaks detected")

# ── 5. Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 10))

# Residuals with break points
axes[0].plot(resid_array, color='steelblue', linewidth=0.8)
axes[0].axvline(int(best_chow.break_point), color='red',    linestyle='--', label=f'Chow break (idx {int(best_chow.break_point)})')
for bp in bp_list:
    axes[0].axvline(bp, color='orange', linestyle=':', label=f'Bai-Perron break (idx {bp})')
axes[0].set_title("Residuals with Structural Breaks")
axes[0].legend()

# Chow F-stat across all candidate break points
axes[1].plot(chow_df['break_point'], chow_df['F_stat'], color='tomato', linewidth=0.9)
axes[1].axvline(int(best_chow.break_point), color='red', linestyle='--')
axes[1].set_title("Chow F-statistic Across Candidate Break Points")
axes[1].set_xlabel("Break Point Index")
axes[1].set_ylabel("F-statistic")

# Distribution of GARCH std residuals vs normal
x = np.linspace(std_resid.min(), std_resid.max(), 200)
axes[2].hist(std_resid, bins=40, density=True, color='steelblue', alpha=0.6, label='Std residuals')
axes[2].plot(x, stats.norm.pdf(x), 'r-',  linewidth=2, label='Normal')
axes[2].plot(x, stats.t.pdf(x, df=5),  'g--', linewidth=2, label='t(df=5)')
axes[2].set_title(f"GARCH Std Residual Distribution  (excess kurtosis = {kurt:.3f})")
axes[2].legend()

plt.tight_layout()
plt.show()

# ── 6. Final Recommendations ─────────────────────────────────────────────────
print("\n── Final Recommendations ──")
if kurt > 1.0:
    print("→ Heavy tails: refit GARCH with dist='t' or dist='skewt'")
if best_chow.p_value < 0.05 or bp_list:
    print("→ Structural break present: consider regime-switching GARCH or split the series")
if kurt <= 1.0 and best_chow.p_value >= 0.05 and not bp_list:
    print("→ No issues: standard GARCH with normal errors is appropriate")

## GARCH-X

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests, adfuller, kpss
from scipy.stats import pearsonr, spearmanr

# --- your inputs here ---
# sq_resid  = np.array(residuals) ** 2   ← realized volatility proxy
# covariate = pd.Series(...)             ← external variable (same length, aligned)

sq_resid  = np.array(residuals) ** 2
covariate = pd.Series(your_covariate)    # replace this

# ── 1. Stationarity of covariate ─────────────────────────────────────────────
def stationarity_check(series, name):
    adf_s, adf_p, *_ = adfuller(series.dropna(), autolag='AIC')
    kpss_s, kpss_p, *_ = kpss(series.dropna(), regression='c', nlags='auto')
    print(f"\n── Stationarity: {name} ──")
    print(f"ADF  p = {adf_p:.4f}  → {'stationary *' if adf_p < 0.05 else 'unit root'}")
    print(f"KPSS p = {kpss_p:.4f}  → {'unit root' if kpss_p < 0.05 else 'stationary *'}")
    is_stationary = adf_p < 0.05 and kpss_p >= 0.05
    return is_stationary

cov_stationary = stationarity_check(covariate, "Covariate")

# Difference if non-stationary
cov_use = covariate if cov_stationary else covariate.diff().dropna()
if not cov_stationary:
    print("→ Covariate differenced to achieve stationarity")
    stationarity_check(cov_use, "Covariate (differenced)")

# ── 2. Align lengths ──────────────────────────────────────────────────────────
min_len  = min(len(sq_resid), len(cov_use))
sq_use   = sq_resid[-min_len:]
cov_arr  = np.array(cov_use)[-min_len:]

# ── 3. Correlation with realized volatility ───────────────────────────────────
r_pearson,  p_pearson  = pearsonr(cov_arr,  sq_use)
r_spearman, p_spearman = spearmanr(cov_arr, sq_use)

print("\n── Correlation: Covariate vs Squared Residuals ──")
print(f"Pearson  r = {r_pearson:.4f},  p = {p_pearson:.4f}  {'*' if p_pearson  < 0.05 else ''}")
print(f"Spearman r = {r_spearman:.4f},  p = {p_spearman:.4f}  {'*' if p_spearman < 0.05 else ''}")

# ── 4. Granger Causality (covariate → squared residuals) ─────────────────────
max_lags = 4
df_gc    = pd.DataFrame({'sq_resid': sq_use, 'covariate': cov_arr})

print(f"\n── Granger Causality: covariate → squared residuals (max lag={max_lags}) ──")
gc_results = grangercausalitytests(df_gc[['sq_resid', 'covariate']], maxlag=max_lags, verbose=False)

print(f"{'Lag':<6} {'F-stat':>10} {'p-value':>10}")
print("-" * 30)
any_sig = False
for lag, result in gc_results.items():
    f_stat = result[0]['ssr_ftest'][0]
    p_val  = result[0]['ssr_ftest'][1]
    sig    = " *" if p_val < 0.05 else ""
    print(f"{lag:<6} {f_stat:>10.4f} {p_val:>10.4f}{sig}")
    if p_val < 0.05:
        any_sig = True

# ── 5. Decision ───────────────────────────────────────────────────────────────
print("\n── Decision ──")
if any_sig:
    print("→ Granger causality detected → covariate can be included in GARCH-X")
else:
    print("→ No Granger causality → covariate does not improve variance model")

if abs(r_spearman) > 0.3 and p_spearman < 0.05:
    print("→ Meaningful correlation with realized volatility → supports inclusion")
else:
    print("→ Weak correlation → covariate may not add much")